# 05 - Finalize, validate & package the Caravan bundle for Zenodo

Run **after `04_caravan_part2_postprocess`**. End to end it makes `04_caravan_bundle` match the
[Caravan "Sharing New Data"](https://github.com/kratzert/Caravan/wiki/Sharing-New-Data) layout and produces the upload zip:

1. Refresh `attributes_additional_ausvic.csv` from the bundle's own timeseries (periods always match the data).
2. Write `licenses/ausvic/license_ausvic.md` + a dated `README.md` (editable sources live in `notebooks/05_assets/`).
3. Copy the simplified shapefile into the bundle.
4. Validate the full structure (4 required folders, each holding only `ausvic/`; 6-col `attributes_other`; gauge_id consistency; no negative/>150 streamflow; shapefile gauge_id-only EPSG:4326) with a hard assert.
5. Zip into `05_zenodo/caravan_ausvic.zip` - ready to upload.

Idempotent; safe to re-run.

In [ ]:
from pathlib import Path
import pandas as pd, geopandas as gpd, glob, os, shutil, zipfile

ROOT          = Path('C:/Users/leela/FloodHubMaribyrnong')
BUNDLE_DIR    = ROOT / '04_caravan_bundle'
SHAPEFILE_DIR = ROOT / '01_shapefile' / 'outputs'
ASSETS_DIR    = ROOT / 'notebooks' / '05_assets'        # editable license + README sources
ZENODO_DIR    = ROOT / '05_zenodo'                      # the upload zip lands here
PREFIX        = 'ausvic'
ZIP_NAME      = 'caravan_ausvic'                        # zip filename + root folder inside it

ATTR_DIR = BUNDLE_DIR / 'attributes' / PREFIX
TS_DIR   = BUNDLE_DIR / 'timeseries' / 'csv' / PREFIX
print('bundle :', BUNDLE_DIR)
print('zip out:', ZENODO_DIR / f'{ZIP_NAME}.zip')

## 1 - Refresh `attributes_additional` from the bundle timeseries

In [ ]:
GAUGE_META = {
    'ausvic_230119': ('Deep Creek',        'bom',     ''),
    'ausvic_230100': ('Deep Creek',        'bom',     ''),
    'ausvic_230107': ('Deep Creek',        'bom',     'Mixed BoM quality (~51% quality-A, ~34% quality-E).'),
    'ausvic_230102': ('Deep Creek',        'bom',     ''),
    'ausvic_230237': ('Maribyrnong River', 'bom',     ''),
    'ausvic_230106': ('Maribyrnong River', 'bom',     'Tidal site: BoM certifies discharge in the flood range (quality-A, incl. Oct-2022 peak); tidal baseflow recorded as uncertified ~0. Reliable flood-event record, low-confidence baseline.'),
    'ausvic_230211': ('Bolinda Creek',     'bom',     ''),
    'ausvic_230200': ('Maribyrnong River', 'hydstra', 'Gauge record starts 1908; merged series begins 1950 (ERA5-Land start).'),
    'ausvic_230206': ('Jackson Creek',     'hydstra', ''),
    'ausvic_230202': ('Jackson Creek',     'hydstra', 'Hydstra record from 1908 but pre-1960 are level-derived artefacts (no rating); kept from 1960.'),
}
SRC = {'bom':     'Bureau of Meteorology, Water Data Online (Water Course Discharge, DMQaQc.Merged; sub-daily -> daily mean)',
       'hydstra': 'Victorian Government Water Measurement Information System (data.water.vic.gov.au; daily mean discharge, var 141.00)'}
DEF_NOTE = {'bom':     'BoM sub-daily discharge aggregated to a time-weighted daily mean; near-zero negative sensor values clipped to 0.',
            'hydstra': 'Victorian Water daily mean discharge (var 141.00); near-zero negatives clipped to 0.'}

rows = []
for gid, (basin, prov, note) in GAUGE_META.items():
    s = pd.to_numeric(pd.read_csv(TS_DIR / f'{gid}.csv', parse_dates=['date']).set_index('date')['streamflow'], errors='coerce')
    fv, lv = s.first_valid_index(), s.last_valid_index()
    period_days = (lv - fv).days + 1
    valid = int(s.loc[fv:lv].notna().sum())
    rows.append({'gauge_id': gid, 'basin_name': basin, 'unit_area': 'km2',
                 'streamflow_period': f'{fv.date()}/{lv.date()}', 'streamflow_missing': round(1 - valid / period_days, 4),
                 'streamflow_units': 'mm/d', 'source': SRC[prov], 'license': 'CC-BY-4.0',
                 'note': note or DEF_NOTE[prov]})
add = pd.DataFrame(rows).set_index('gauge_id').sort_index()
add.to_csv(ATTR_DIR / f'attributes_additional_{PREFIX}.csv', encoding='utf-8-sig')
print('attributes_additional refreshed ->', add.shape)
print(add[['basin_name', 'streamflow_period', 'streamflow_missing']].to_string())

## 2 - Write `licenses/ausvic/license_ausvic.md` + dated `README.md`

In [ ]:
# 2a. license -> licenses/ausvic/license_ausvic.md  (verbatim from the editable source)
lic_dir = BUNDLE_DIR / 'licenses' / PREFIX
lic_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(ASSETS_DIR / f'license_{PREFIX}.md', lic_dir / f'license_{PREFIX}.md')

# 2b. README.md  (fill the timeseries period from the actual bundle data)
dmin = dmax = None
for p in glob.glob(str(TS_DIR / '*.csv')):
    d = pd.to_datetime(pd.read_csv(p, usecols=['date'])['date'])
    dmin = d.min() if dmin is None else min(dmin, d.min())
    dmax = d.max() if dmax is None else max(dmax, d.max())
readme = (ASSETS_DIR / 'README_template.md').read_text(encoding='utf-8')
readme = readme.replace('__PSTART__', str(dmin.date())).replace('__PEND__', str(dmax.date()))
(BUNDLE_DIR / 'README.md').write_text(readme, encoding='utf-8')

# 2c. drop superseded / stray items so the bundle matches the Caravan layout exactly
(BUNDLE_DIR / 'LICENSE.txt').unlink(missing_ok=True)
shutil.rmtree(BUNDLE_DIR / 'temp', ignore_errors=True)
print('license ->', lic_dir / f'license_{PREFIX}.md')
print('README period:', dmin.date(), '->', dmax.date())

## 3 - Copy the simplified shapefile into the bundle

In [ ]:
sd = BUNDLE_DIR / 'shapefiles' / PREFIX
sd.mkdir(parents=True, exist_ok=True)
for ext in ['shp', 'shx', 'dbf', 'prj', 'cpg']:
    src = SHAPEFILE_DIR / f'{PREFIX}_basin_shapes.{ext}'
    if src.is_file():
        shutil.copy(src, sd / src.name)
print('shapefile siblings in bundle:', sorted(p.name for p in sd.glob(f'{PREFIX}_basin_shapes.*')))

## 4 - Validate the bundle against the Caravan layout

In [ ]:
def ids(p): return set(pd.read_csv(p)['gauge_id'])
def names(d): return sorted(x.name for x in d.iterdir() if not x.name.startswith('.'))
shp_path = BUNDLE_DIR / 'shapefiles' / PREFIX / f'{PREFIX}_basin_shapes.shp'
checks = {}

for f in ['attributes_other', 'attributes_hydroatlas', 'attributes_caravan', 'attributes_additional']:
    checks[f'file: {f}'] = (ATTR_DIR / f'{f}_{PREFIX}.csv').is_file()
checks['file: shapefile'] = shp_path.is_file()
checks['file: license_ausvic.md'] = (BUNDLE_DIR / 'licenses' / PREFIX / f'license_{PREFIX}.md').is_file()

# Caravan structure: each top folder holds the {prefix} subfolder and nothing else
checks['attributes/ = [ausvic]'] = names(BUNDLE_DIR / 'attributes') == [PREFIX]
checks['shapefiles/ = [ausvic]'] = names(BUNDLE_DIR / 'shapefiles') == [PREFIX]
checks['licenses/ = [ausvic]']   = names(BUNDLE_DIR / 'licenses') == [PREFIX]
checks['timeseries/ = [csv, netcdf]']   = names(BUNDLE_DIR / 'timeseries') == ['csv', 'netcdf']
checks['timeseries/csv/ = [ausvic]']    = names(BUNDLE_DIR / 'timeseries' / 'csv') == [PREFIX]
checks['timeseries/netcdf/ = [ausvic]'] = names(BUNDLE_DIR / 'timeseries' / 'netcdf') == [PREFIX]

ao = pd.read_csv(ATTR_DIR / f'attributes_other_{PREFIX}.csv')
checks['attributes_other = 6 standard cols'] = set(ao.columns) == {'gauge_id', 'gauge_name', 'country', 'gauge_lat', 'gauge_lon', 'area'}

base = set(ao['gauge_id'])
for f in ['attributes_hydroatlas', 'attributes_caravan', 'attributes_additional']:
    checks[f'gauge_id match: {f}'] = ids(ATTR_DIR / f'{f}_{PREFIX}.csv') == base
ts = set(os.path.basename(p)[:-4] for p in glob.glob(str(TS_DIR / '*.csv')))
nc = set(os.path.basename(p)[:-3] for p in glob.glob(str(BUNDLE_DIR / 'timeseries' / 'netcdf' / PREFIX / '*.nc')))
g = gpd.read_file(shp_path)
checks['gauge_id match: timeseries/csv']    = ts == base
checks['gauge_id match: timeseries/netcdf'] = nc == base
checks['gauge_id match: shapefile']         = set(g['gauge_id']) == base

neg = hi = 0
for p in glob.glob(str(TS_DIR / '*.csv')):
    v = pd.to_numeric(pd.read_csv(p)['streamflow'], errors='coerce'); neg += int((v < 0).sum()); hi += int((v > 150).sum())
checks['no negative streamflow'] = neg == 0
checks['no >150 mm/d streamflow'] = hi == 0
checks['shapefile gauge_id-only'] = list(g.columns) == ['gauge_id', 'geometry']
checks['shapefile EPSG:4326'] = (g.crs is not None and g.crs.to_epsg() == 4326)

print('BUNDLE VALIDATION'); print('=' * 52)
for k, v in checks.items():
    print(f'  {"PASS" if v else "FAIL"}  {k}')
bad = [k for k, v in checks.items() if not v]
assert not bad, 'VALIDATION FAILED: ' + ', '.join(bad)
print(f'\nAll {len(checks)} checks passed.')

## 5 - Package the Zenodo upload zip

In [ ]:
ZENODO_DIR.mkdir(parents=True, exist_ok=True)
zip_path = ZENODO_DIR / f'{ZIP_NAME}.zip'
if zip_path.exists():
    zip_path.unlink()

include_dirs  = ['attributes', 'timeseries', 'shapefiles', 'licenses']   # the 4 required Caravan folders
include_files = ['README.md']
n = 0
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in include_files:
        if (BUNDLE_DIR / f).is_file():
            z.write(BUNDLE_DIR / f, f'{ZIP_NAME}/{f}'); n += 1
    for d in include_dirs:
        for p in sorted((BUNDLE_DIR / d).rglob('*')):
            if p.is_file():
                z.write(p, f'{ZIP_NAME}/' + p.relative_to(BUNDLE_DIR).as_posix()); n += 1

mb = zip_path.stat().st_size / 1e6
with zipfile.ZipFile(zip_path) as z:
    top = sorted({e.split('/')[1] for e in z.namelist() if e.count('/') >= 1})
print(f'wrote {zip_path}')
print(f'  {n} files, {mb:.1f} MB')
print(f'  zip root: {ZIP_NAME}/')
print(f'  contains: {", ".join(top)}')
print('\nReady to upload to Zenodo.')